# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [13]:
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import ChatOpenAI
from langchain import HuggingFaceHub
from langchain_core.runnables import Runnable
from langchain_core.prompts import PromptTemplate
from langchain.chains import SimpleSequentialChain, LLMChain, LLMMathChain, TransformChain, SequentialChain
from langchain_core.output_parsers import StrOutputParser
from langchain import FewShotPromptTemplate
from langchain.callbacks import get_openai_callback
import inspect
import re
import os

In [16]:

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')


In [3]:
!pip install pandas

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.4.6 which is incompatible.
langchain-community 0.2.19 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.4.6 which is incompatible.


In [4]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [5]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [6]:
!pip install langchain_community

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 3.1.0 requires numpy>=2.0.2; python_version < "3.14", but you have numpy 1.26.4 which is incompatible.


In [7]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [17]:
#Replace None by your own value and justify
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)


In [18]:
prompt = ChatPromptTemplate.from_template(
    "Write a concise, appealing description for this product: {product}"
)

In [19]:

chain = prompt | llm 

In [11]:
!pip install langchain-openai openai


In [20]:
product ={"product": "Queen Size Sheet Set"}
chain.invoke(product)

AIMessage(content='Refresh your bedroom with this queen-size sheet set, designed to bring a polished, inviting feel to your bed. A simple way to make every night feel a little more comfortable.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 20, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 85, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-9fd7cb8d-10d9-417f-a8d7-a9c05c1967f7-0', usage_metadata={'input_tokens': 20, 'output_tokens': 129, 'total_tokens': 149})

## SimpleSequentialChain

In [21]:
from langchain.chains import SimpleSequentialChain

In [32]:
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
  "What is the best name to describe a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [33]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
   "Write a 20-word description for the following company: {company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [34]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [35]:
result = overall_simple_chain.invoke(product["product"])
print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The best general term is **bedding manufacturer**. More specifically, it’s a **home-textile manufacturer specializing in queen-size sheet sets**.
A bedding manufacturer specializing in queen-size sheet sets, creating quality home textiles that bring comfort, durability, and style to bedrooms.

> Finished chain.
A bedding manufacturer specializing in queen-size sheet sets, creating quality home textiles that bring comfort, durability, and style to bedrooms.


In [36]:
result = overall_simple_chain.invoke("Wireless Noise-Cancelling Headphones")
print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The best general description is **an audio technology company specializing in wireless noise-cancelling headphones**.

If you mean a catchy brand name for the company, I can suggest some options.
An audio technology company specializing in wireless noise-cancelling headphones, delivering immersive sound, comfort, and freedom for music lovers and travelers.

> Finished chain.
An audio technology company specializing in wireless noise-cancelling headphones, delivering immersive sound, comfort, and freedom for music lovers and travelers.


In [37]:
result = overall_simple_chain.invoke("Stainless Steel Water Bottle")
print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The clearest description is **“stainless steel water bottle manufacturer.”** More broadly, you could call it a **“reusable drinkware manufacturer.”**

If you mean a catchy brand name for the company, tell me the style you want—such as modern, eco-friendly, or premium.
A stainless steel water bottle manufacturer creating reusable drinkware that offers an alternative to disposable beverage containers for everyday hydration.

> Finished chain.
A stainless steel water bottle manufacturer creating reusable drinkware that offers an alternative to disposable beverage containers for everyday hydration.


**Repeat the above twice for different products**

## SequentialChain

In [ ]:
from langchain.chains import SequentialChain

In [38]:
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)


first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review into German:\n\n{Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="German_Review"
                    )

In [39]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{German_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )

In [40]:
# prompt template 3: detect the review's language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review written in? Answer with just the language name:\n\n{Review}"
)

# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

In [41]:

# prompt template 4: follow up message that takes the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [42]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["German_Review", "summary", "language", "followup_message"],
    verbose=True
)

In [43]:
review = df.Review[5]
overall_chain.invoke({"Review": review})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'German_Review': 'Ich finde den Geschmack mittelmäßig. Der Schaum hält nicht, das ist seltsam. Ich kaufe dieselben im Handel, und sie schmecken viel besser … Alte Charge oder Fälschung!?',
 'summary': 'The reviewer finds the taste mediocre and the foam unusually short-lived compared with store-bought versions, raising concerns about an old batch or a counterfeit.',
 'language': 'French',
 'followup_message': 'Nous sommes désolés que le goût et la tenue de la mousse n’aient pas été à la hauteur de vos attentes. Une mousse qui retombe rapidement peut dépendre de la préparation, mais nous prenons aussi au sérieux votre inquiétude concernant le lot ou l’authenticité du produit.\n\nPour que nous puissions vérifier, pourriez-vous nous envoyer une photo de l’emballage avec le numéro de lot et la date de péremption,

**Repeat the above twice for different products or reviews**

In [44]:
reviews = [
    "Las sábanas son muy suaves, pero después de dos lavados empezaron a desteñirse.",
    "Questo frullatore è potentissimo e facile da pulire. Lo consiglio a tutti!"
]

for r in reviews:
    result = overall_chain.invoke({"Review": r})
    print(f"Language: {result['language']}")
    print(f"Summary: {result['summary']}")
    print(f"Follow-up: {result['followup_message']}\n")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
Language: Spanish
Summary: The bedsheets are very soft but began to fade after just two washes.
Follow-up: Gracias por compartir tu experiencia. Nos alegra que las sábanas te parezcan muy suaves, pero lamentamos que hayan comenzado a desteñirse después de solo dos lavados. No es la calidad que esperamos ofrecer. Por favor, contáctanos para que podamos ayudarte a resolverlo.


> Finished chain.
Language: Italiano
Summary: The reviewer highly recommends this powerful blender, noting that it’s easy to clean.
Follow-up: Grazie per aver condiviso la tua esperienza! Siamo felici che tu apprezzi la potenza del frullatore e che sia facile da pulire. Buoni frullati!



## Router Chain

In [45]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [46]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [47]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [48]:
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)

In [49]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [50]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [52]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [53]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [54]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [55]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black-body radiation is the electromagnetic radiation emitted by an ideal object that absorbs all radiation falling on it. Its spectrum depends only on the object’s temperature: hotter objects emit more energy and peak at shorter wavelengths. A glowing-hot piece of metal is an everyday approximation to a black body.'

In [56]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'2 + 2 = 4.'

In [57]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why do nearly all cells in our body contain DNA, and are there exceptions?'}
> Finished chain.


'Nearly all cells contain DNA because they descend from the fertilized egg, whose DNA is copied as cells divide. That DNA provides instructions a cell can use to make proteins and maintain itself. Most cells in one person carry essentially the same genome; they become different kinds of cells mainly by switching different genes on or off.\n\nThere are exceptions:\n\n- **Mature red blood cells** in humans and other mammals expel their nucleus as they develop. They also lose their mitochondria, so mature red blood cells have no DNA.\n- **Some mature lens cells** and the outermost **cornified skin cells** also lose their nuclei as they specialize.\n- **Platelets** are fragments shed from larger cells, not complete cells. They have no nucleus, though they do contain mitochondria and mitochondrial DNA.\n\nOther cells may have unusual amounts or arrangements of DNA—for example, some have multiple nuclei—but they still contain DNA. These exceptions are specialized cells whose jobs can be done

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [58]:
chef_template = """You are a professional chef. You give simple, \
practical cooking advice with clear steps.

Here is a question:
{input}"""

poet_template = """You are a creative poet. You answer every \
question in the form of a short, rhyming poem.

Here is a question:
{input}"""

new_prompt_infos = prompt_infos + [
    {"name": "cooking", "description": "Good for answering cooking and recipe questions", "prompt_template": chef_template},
    {"name": "poetry", "description": "Good for requests to write poems or creative verse", "prompt_template": poet_template},
]

In [59]:
new_destination_chains = {}
for p_info in new_prompt_infos:
    prompt = ChatPromptTemplate.from_template(template=p_info["prompt_template"])
    new_destination_chains[p_info["name"]] = LLMChain(llm=llm, prompt=prompt)

In [60]:
# Rebuild the router with the new list of destinations
new_destinations_str = "\n".join(f"{p['name']}: {p['description']}" for p in new_prompt_infos)


In [61]:
new_router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=new_destinations_str)
new_router_prompt = PromptTemplate(
    template=new_router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

In [62]:
new_router_chain = LLMRouterChain.from_llm(llm, new_router_prompt)

new_chain = MultiPromptChain(
    router_chain=new_router_chain,
    destination_chains=new_destination_chains,
    default_chain=default_chain,
    verbose=False
)

In [ ]:

new_chain.invoke({"input": "What is Newton's second law?"})          # physics

{'input': "What is Newton's second law?",
 'text': 'Newton’s second law says that an object’s acceleration depends on the net force acting on it and its mass:\n\n\\[\n\\vec F_{\\text{net}} = m\\vec a\n\\]\n\nSo, more force causes more acceleration, while more mass causes less acceleration for the same force.'}

In [64]:
new_chain.invoke({"input": "How do I make fluffy pancakes?"})        # cooking


{'input': 'How do I make fluffy pancakes?',
 'text': '### Fluffy pancakes\n\n**Makes about 8 pancakes**\n\n**Ingredients**\n- 1½ cups all-purpose flour\n- 2 tablespoons sugar\n- 2 teaspoons baking powder\n- ½ teaspoon baking soda\n- ¼ teaspoon salt\n- 1¼ cups milk\n- 1 egg\n- 3 tablespoons melted butter, plus more for the pan\n\n**Steps**\n1. Whisk the flour, sugar, baking powder, baking soda, and salt in a bowl.\n2. In another bowl, whisk the milk, egg, and melted butter. Pour into the dry ingredients and stir just until combined. A few lumps are fine—overmixing makes pancakes tough.\n3. Let the batter rest for 5–10 minutes while a lightly buttered pan heats over **medium-low** heat.\n4. Pour about ¼ cup batter per pancake. Cook until bubbles form on top and the edges look set, about 2–3 minutes. Flip once and cook another 1–2 minutes.\n5. Serve warm.\n\nFor extra lift, check that your baking powder is fresh, and don’t press down on the pancakes after flipping.'}

In [65]:
new_chain.invoke({"input": "Write a poem about the ocean at night"}) # poetry

{'input': 'Write a poem about the ocean at night',
 'text': 'The moon lays silver on the sea,  \nWhile waves whisper secrets to the shore;  \nStars drift like lanterns, wild and free,  \nAnd darkness hums an endless ocean song once more.'}